# Practica - Clase 1: Introduccion a la IA avanzada y tu primera red neuronal

## Que vas a practicar

En esta practica aplicas los dos mecanismos que sostienen todo el aprendizaje profundo (*deep learning*): el calculo del gradiente mediante la regla de la cadena (*backpropagation*) y el entrenamiento de una red con PyTorch. Resuelves dos ejercicios:

1. Completas a mano el calculo del gradiente de una neurona con un solo peso, sin ninguna libreria de diferenciacion automatica.
2. Construyes y entrenas un perceptron multicapa (MLP) en PyTorch para clasificar un conjunto de datos no lineal, y visualizas la frontera de decision que aprende.

## Requisito previo

Antes de empezar, repasa la clase 1 en directo y los capitulos 1 y 2 del manual ("Introduccion al deep learning avanzado" y "Fundamentos de las redes neuronales"). Necesitas entender que es una neurona artificial, que hace una funcion de activacion y que papel juega el descenso de gradiente.

## Como trabajar este cuaderno

Cada ejercicio sigue el mismo patron: lees el enunciado, completas las partes marcadas con `# TODO` en el codigo y ejecutas la celda. Abre la solucion colapsada solo si te atascas o quieres comprobar tu respuesta; intenta resolver cada ejercicio por tu cuenta antes de mirarla.

Abre este cuaderno en Google Colab y empieza por el ejercicio 1.

## Ejercicio 1 - Backpropagation a mano: el gradiente de una sola neurona

**Contexto.** En DIH DATAlife trabajamos con sensores del internet de las cosas (IoT) instalados en explotaciones agricolas y forestales, que miden variables como la humedad del suelo. Imagina un sensor que entrega la humedad normalizada del suelo (`x`, entre 0 y 1) a una neurona que decide con que probabilidad hay que activar el riego. Esa neurona tiene un unico peso `w` (sin sesgo, para mantener el ejemplo simple), usa la activacion sigmoide y se entrena comparando su prediccion con un objetivo mediante la perdida cuadratica (error cuadratico medio, MSE).

$$\hat{y} = \sigma(w \cdot x), \qquad \mathcal{L} = (\hat{y} - y_{objetivo})^2$$

En la celda de codigo siguiente ya tienes escritos los datos, el peso inicial, las funciones auxiliares (`sigmoide` y `derivada_sigmoide`) y el pase hacia adelante (*forward pass*) dentro del bucle de entrenamiento. Te falta completar el pase hacia atras (*backward pass*): el calculo del gradiente de la perdida respecto al peso `w`, aplicando la regla de la cadena termino a termino, y la actualizacion del peso con descenso de gradiente.

**Pistas:**

- La regla de la cadena descompone el gradiente en tres derivadas que se multiplican: `dL/dw = dL/dy_pred * dy_pred/dz * dz/dw`
- `dL/dy_pred` es la derivada de la perdida cuadratica respecto a la prediccion: `2 * (y_pred - y_objetivo)`
- `dy_pred/dz` es la derivada de la sigmoide en `z`; ya tienes la funcion `derivada_sigmoide` escrita para calcularla
- `dz/dw` es la derivada de `z = w * x` respecto a `w`, que es simplemente `x`
- La actualizacion de descenso de gradiente es: `w = w - eta * gradiente_w`

In [ ]:
import numpy as np

# --- Datos del sensor ---
# x: humedad normalizada del suelo (0 = seco, 1 = saturado)
# y_objetivo: probabilidad ideal de activar el riego para esa humedad
x = 2.0
y_objetivo = 1.0

# Peso inicial de la neurona (en un caso real empezaria en un valor aleatorio;
# aqui lo fijamos para que puedas verificar tus calculos a mano)
w = 0.3

# Tasa de aprendizaje (learning rate)
eta = 0.1

def sigmoide(z):
    """Funcion de activacion sigmoide: comprime cualquier numero real en (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))

def derivada_sigmoide(z):
    """Derivada de la sigmoide: f'(z) = f(z) * (1 - f(z))."""
    s = sigmoide(z)
    return s * (1.0 - s)

historial_perdida = []

for paso in range(5):
    # --- Pase hacia adelante (forward pass), ya escrito ---
    z = w * x
    y_pred = sigmoide(z)
    perdida = (y_pred - y_objetivo) ** 2
    historial_perdida.append(perdida)

    # --- Pase hacia atras (backward pass): completa los TODO ---

    # TODO 1: calcula dL/dy_pred, la derivada de la perdida cuadratica respecto a y_pred
    dL_dypred = ...

    # TODO 2: calcula dy_pred/dz, la derivada de la sigmoide en z (usa derivada_sigmoide)
    dypred_dz = ...

    # TODO 3: calcula dz/dw, la derivada de z = w * x respecto a w
    dz_dw = ...

    # TODO 4: combina las tres derivadas anteriores con la regla de la cadena
    # para obtener el gradiente de la perdida respecto al peso w
    gradiente_w = ...

    # TODO 5: actualiza w con descenso de gradiente (w = w - eta * gradiente)
    w = ...

    print(f"paso {paso}: w={w:.5f}  y_pred={y_pred:.5f}  perdida={perdida:.5f}  gradiente={gradiente_w:.5f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(historial_perdida, marker="o")
plt.xlabel("Paso de entrenamiento")
plt.ylabel("Perdida")
plt.title("Evolucion de la perdida durante el entrenamiento a mano")
plt.show()

<details>
<summary>💡 Ver solucion</summary>

```python
import numpy as np

x = 2.0
y_objetivo = 1.0
w = 0.3
eta = 0.1

def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))

def derivada_sigmoide(z):
    s = sigmoide(z)
    return s * (1.0 - s)

historial_perdida = []

for paso in range(5):
    # --- Pase hacia adelante ---
    z = w * x
    y_pred = sigmoide(z)
    perdida = (y_pred - y_objetivo) ** 2
    historial_perdida.append(perdida)

    # --- Pase hacia atras: regla de la cadena, termino a termino ---
    dL_dypred = 2 * (y_pred - y_objetivo)   # derivada de la perdida respecto a y_pred
    dypred_dz = derivada_sigmoide(z)        # derivada de la sigmoide respecto a z
    dz_dw = x                               # derivada de z = w * x respecto a w

    # dL/dw = dL/dy_pred * dy_pred/dz * dz/dw
    gradiente_w = dL_dypred * dypred_dz * dz_dw

    # --- Descenso de gradiente ---
    w = w - eta * gradiente_w

    print(f"paso {paso}: w={w:.5f}  y_pred={y_pred:.5f}  perdida={perdida:.5f}  gradiente={gradiente_w:.5f}")
```

Si ejecutas esta solucion, veras que `y_pred` se acerca poco a poco a `1.0` y que la perdida decrece en cada paso: eso confirma que el gradiente que calculaste apunta en la direccion correcta para minimizar el error. Cuando mas adelante entrenes redes con miles de pesos, PyTorch calculara automaticamente esta misma cadena de derivadas con `.backward()`.
</details>

## Ejercicio 2 - Tu primer perceptron multicapa (MLP) con PyTorch

**Contexto.** Quieres construir un clasificador binario capaz de separar dos clases que no se pueden trazar con una sola linea recta. Vas a usar el conjunto de datos `make_moons` de scikit-learn, que genera dos "medias lunas" entrelazadas: un ejemplo no lineal, emparentado en espiritu con el problema XOR (o exclusivo) que viste en la clase 1.

Vas a construir un perceptron multicapa (MLP), la arquitectura de red neuronal mas sencilla con capas ocultas, y a entrenarlo con PyTorch. Ya tienes escritos la carga y preparacion de los datos, y el bucle de entrenamiento completo (pase hacia adelante, calculo de la perdida, pase hacia atras y actualizacion de los pesos). Te falta definir la arquitectura del modelo: las capas que lo componen y como se encadenan en el metodo `forward`.

**Pistas:**

- Una capa lineal en PyTorch se crea con `nn.Linear(n_entradas, n_salidas)`
- Entre capas necesitas una funcion de activacion no lineal; usa `nn.ReLU()`, como viste en el capitulo 2
- Como es un problema de clasificacion binaria (dos clases), la capa final produce una sola salida, que conviertes en probabilidad con `nn.Sigmoid()`
- En el metodo `forward`, encadena las capas en el mismo orden en el que las definiste en `__init__`: entrada -> capa oculta -> activacion -> capa de salida -> sigmoide

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

# --- Generamos el dataset de las dos medias lunas ---
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convertimos los datos a tensores de PyTorch
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Visualizamos los datos de entrenamiento
plt.figure(figsize=(5, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap="coolwarm", edgecolors="k")
plt.title("Dataset make_moons")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [ ]:
class MLP(nn.Module):
    """Perceptron multicapa con una capa oculta para clasificacion binaria."""

    def __init__(self, n_entradas=2, n_ocultas=16):
        super().__init__()
        # TODO: define las cuatro piezas del modelo
        # - self.capa1: capa lineal de n_entradas a n_ocultas
        # - self.activacion: una activacion ReLU
        # - self.capa2: capa lineal de n_ocultas a 1 (una sola salida, clasificacion binaria)
        # - self.sigmoide: activacion sigmoide para convertir la salida en probabilidad
        self.capa1 = ...
        self.activacion = ...
        self.capa2 = ...
        self.sigmoide = ...

    def forward(self, x):
        # TODO: encadena las capas definidas arriba en el orden correcto
        # entrada -> capa1 -> activacion -> capa2 -> sigmoide
        salida = ...
        return salida

modelo = MLP()
print(modelo)

In [ ]:
criterio = nn.BCELoss()  # entropia cruzada binaria, para clasificacion con dos clases
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)

n_epocas = 200
historial_perdida = []

for epoca in range(n_epocas):
    # --- Pase hacia adelante ---
    y_pred = modelo(X_train)
    perdida = criterio(y_pred, y_train)
    historial_perdida.append(perdida.item())

    # --- Pase hacia atras y actualizacion ---
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

    if (epoca + 1) % 20 == 0:
        print(f"epoca {epoca + 1}/{n_epocas}  perdida={perdida.item():.4f}")

plt.plot(historial_perdida)
plt.xlabel("Epoca")
plt.ylabel("Perdida (BCE)")
plt.title("Evolucion de la perdida durante el entrenamiento")
plt.show()

In [ ]:
with torch.no_grad():
    y_pred_test = modelo(X_test)
    y_pred_clase = (y_pred_test > 0.5).float()
    precision = (y_pred_clase == y_test).float().mean()

print(f"Precision (accuracy) en el conjunto de prueba: {precision.item():.2%}")

In [ ]:
def graficar_frontera_decision(modelo, X, y):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

    malla = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        Z = modelo(malla)
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, levels=20, cmap="coolwarm", alpha=0.7)
    plt.scatter(X[:, 0], X[:, 1], c=y.squeeze(), cmap="coolwarm", edgecolors="k")
    plt.title("Frontera de decision aprendida por el MLP")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

graficar_frontera_decision(modelo, X_test, y_test)

<details>
<summary>💡 Ver solucion</summary>

```python
class MLP(nn.Module):
    """Perceptron multicapa con una capa oculta para clasificacion binaria."""

    def __init__(self, n_entradas=2, n_ocultas=16):
        super().__init__()
        self.capa1 = nn.Linear(n_entradas, n_ocultas)  # entrada -> capa oculta
        self.activacion = nn.ReLU()                    # no linealidad entre capas
        self.capa2 = nn.Linear(n_ocultas, 1)            # capa oculta -> salida (1 neurona)
        self.sigmoide = nn.Sigmoid()                    # convierte la salida en probabilidad

    def forward(self, x):
        x = self.capa1(x)
        x = self.activacion(x)
        x = self.capa2(x)
        salida = self.sigmoide(x)
        return salida

modelo = MLP()
print(modelo)
```

Fijate en el patron: la capa oculta (`capa1` + `activacion`) deforma el espacio de entrada hasta volverlo separable, tal como viste en el capitulo 2 con el ejemplo del XOR; la capa de salida (`capa2` + `sigmoide`) solo necesita trazar una frontera simple sobre ese espacio ya transformado. Si ejecutas el bucle de entrenamiento y el grafico de la frontera de decision con este modelo, deberias ver una curva que separa las dos medias lunas con una precision cercana o superior al 95 % en el conjunto de prueba.
</details>

## Has completado la practica de la clase 1

Acabas de tocar, con tus propias manos, las dos caras de una misma moneda. En el ejercicio 1 calculaste un gradiente termino a termino, sin ninguna magia, solo la regla de la cadena aplicada a una neurona. En el ejercicio 2 viste como PyTorch automatiza exactamente ese mismo calculo, por complejo que sea el modelo, gracias a `autograd`, y como una capa oculta con activacion no lineal resuelve un problema que ninguna linea recta puede resolver.

Este patron de tres pasos, pase hacia adelante, `backward()`, `optimizador.step()`, es el que vas a repetir en cada arquitectura que construyas durante el resto del curso: redes convolucionales, redes recurrentes, mecanismos de atencion y modelos generativos parten todos de esta misma base.

**Guarda una copia de este cuaderno en tu Google Drive y repasalo antes de la proxima clase.**